# Document Tables → Snowflake Tables

**Premise:** the schema is never assumed known. Phase 1 parses every document and discovers what tables exist before any extraction is attempted.

**Single-method variant.** Both discovery and extraction use LLM calls against the parsed markdown. The deterministic markdown-parsing alternatives are omitted for clarity, so the document is processed exactly once and every table follows the same route.

| Phase | What it does | Method |
| --- | --- | --- |
| **1 Discover** | Parse to markdown, describe every table found | `AI_PARSE_DOCUMENT` → `AI_COMPLETE` |
| **2 Extract** | One row-oriented extraction per discovered table | `AI_COMPLETE` |
| **3 Validate** | Deterministic assertions, then a narrowly scoped judge | SQL, then `AI_COMPLETE` |

Phase 3 keeps its SQL assertions deliberately: row counts, null checks and header-leakage detection are arithmetic, and SQL does arithmetic exactly and for free. The LLM judge answers only what SQL cannot.

In [ ]:
# Pipeline configuration. Change these, then run the notebook top to bottom.
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# One location for everything: the source documents and the generated tables
# both live in this database and schema.
DB     = "DOCUMENT_EXTRACTION"
SCHEMA = "PDF"
STAGE  = "DOC_STAGE"

FILE_FILTER = "%.pdf"          # which files on the stage to process

# If the discovery cell fails with an unknown-model error, change this first.
MODEL = "claude-sonnet-4-5"

# --- derived ---------------------------------------------------------------
SCHEMA_FQ = f"{DB}.{SCHEMA}"             # qualifies every table and view
STAGE_FQ  = f"{DB}.{SCHEMA}.{STAGE}"     # qualifies the document stage

session.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_FQ}").collect()

print(f"database.schema : {SCHEMA_FQ}")
print(f"stage           : @{STAGE_FQ}")
print(f"file filter     : {FILE_FILTER}")
print(f"model           : {MODEL}")

# Phase 1 drives off DIRECTORY(), which reads the stage's directory table
# rather than the stage itself. A stage can hold files while its directory
# table reports none, so check now instead of discovering it as an empty parse.
n_visible = session.sql(
    f"SELECT COUNT(*) AS n FROM DIRECTORY(@{STAGE_FQ})"
).collect()[0]["N"]
print(f"files visible   : {n_visible}")

if n_visible == 0:
    print(f"\n  WARNING: DIRECTORY() sees no files, so Phase 1 will parse nothing.")
    print(f"  Confirm the stage actually has files:")
    print(f"      LIST @{STAGE_FQ};")
    print(f"  If it does, the directory table needs a refresh:")
    print(f"      ALTER STAGE {STAGE_FQ} REFRESH;")

---

## Phase 1 — Discover

Runs unconditionally. Nothing downstream may assume a schema that did not come from here.

In [ ]:
-- 1a. Parse every matching document once, persist the raw response using AI_PARSE_DOCUMENT().
--     Driving off DIRECTORY() handles any number of files.
CREATE OR REPLACE TABLE {{SCHEMA_FQ}}.PARSED_DOCS AS
SELECT
  RELATIVE_PATH                    AS file_name,
  AI_PARSE_DOCUMENT(
    TO_FILE('@{{STAGE_FQ}}', RELATIVE_PATH),
    {'mode': 'LAYOUT'}
  )                                AS resp,
  CURRENT_TIMESTAMP()              AS parsed_at
FROM DIRECTORY(@{{STAGE_FQ}})
WHERE RELATIVE_PATH ILIKE '{{FILE_FILTER}}'

In [ ]:
select resp from {{SCHEMA_FQ}}.parsed_docs;

In [ ]:
-- 1a. Convenience view over the markdown (aka persisting the markdown content)
-- CREATE OR REPLACE VIEW {{SCHEMA_FQ}}.PARSED_MARKDOWN AS
-- SELECT file_name,
--        resp:content::VARCHAR AS content,
--        parsed_at
-- FROM {{SCHEMA_FQ}}.PARSED_DOCS;

CREATE OR REPLACE VIEW {{SCHEMA_FQ}}.PARSED_MARKDOWN AS
with parsed_docs as (
    SELECT
      RELATIVE_PATH                    AS file_name,
      AI_PARSE_DOCUMENT(
        TO_FILE('@{{STAGE_FQ}}', RELATIVE_PATH),
        {'mode': 'LAYOUT'}
      )                                AS resp,
      CURRENT_TIMESTAMP()              AS parsed_at
    FROM DIRECTORY(@{{STAGE_FQ}})
    WHERE RELATIVE_PATH ILIKE '{{FILE_FILTER}}'
)
SELECT file_name,
       resp:content::VARCHAR AS content,
       parsed_at
FROM parsed_docs;

In [ ]:
select * from {{SCHEMA_FQ}}.parsed_markdown;

In [ ]:
-- 1b. One call per document. Catches 
-- multi-level headers, semantic titles, column types, unit qualifiers.
-- Note the row-oriented response_format: one object per discovered table.
CREATE OR REPLACE TABLE {{SCHEMA_FQ}}.DESCRIBED_TABLES AS
WITH described AS (
  SELECT
    file_name,
    AI_COMPLETE(
      model  => '{{MODEL}}',
      prompt => 'Identify every table in the markdown document below. For each table report: '
             || 'a short snake_case key unique within the document; its title as written; '
             || 'its column headers in left-to-right order; a SQL type for each column '
             || '(VARCHAR, INT, FLOAT or DATE); whether the header is multi-level, meaning a '
             || 'header row whose labels span several columns sits above a second header row; '
             || 'the spanning group labels if so, otherwise an empty array; the number of data '
             || 'rows excluding all header rows; and any unit qualifier such as "in thousands" '
             || 'or a currency, otherwise an empty string. Report tables in document order.'
             || CHR(10) || CHR(10) || content,
      model_parameters => {'temperature': 0},
      response_format  => {
        'type': 'json',
        'schema': {
          'type': 'object',
          'properties': {
            'tables': {
              'type': 'array',
              'items': {
                'type': 'object',
                'properties': {
                  'table_key':          {'type': 'string'},
                  'title':              {'type': 'string'},
                  'columns':            {'type': 'array', 'items': {'type': 'string'}},
                  'column_types':       {'type': 'array', 'items': {'type': 'string'}},
                  'multi_level_header': {'type': 'boolean'},
                  'header_groups':      {'type': 'array', 'items': {'type': 'string'}},
                  'row_count':          {'type': 'integer'},
                  'unit_qualifier':     {'type': 'string'}
                },
                'required': ['table_key','title','columns','column_types',
                             'multi_level_header','row_count']
              }
            }
          },
          'required': ['tables']
        }
      }
    ) AS resp
  FROM {{SCHEMA_FQ}}.PARSED_MARKDOWN
)
SELECT
  file_name,
  t.INDEX + 1                             AS table_ordinal,
  t.VALUE:table_key::VARCHAR              AS table_key,
  t.VALUE:title::VARCHAR                  AS title,
  t.VALUE:columns                         AS columns,
  t.VALUE:column_types                    AS column_types,
  t.VALUE:multi_level_header::BOOLEAN     AS multi_level_header,
  t.VALUE:header_groups                   AS header_groups,
  t.VALUE:row_count::INT                  AS row_count,
  NULLIF(t.VALUE:unit_qualifier::VARCHAR, '') AS unit_qualifier,
  CURRENT_TIMESTAMP()                     AS discovered_at
FROM described,
     LATERAL FLATTEN(input => TRY_PARSE_JSON(resp):tables) t

In [ ]:
select * from {{SCHEMA_FQ}}.DESCRIBED_TABLES;

In [ ]:
select table_ordinal, title, columns, multi_level_header from {{SCHEMA_FQ}}.DESCRIBED_TABLES;

In [ ]:
-- 1c. Registry of discovered schemas. Persisting discovery makes re-extraction
-- cheap and gives Phase 3 the row-count and column-name assertions.
-- response_schema holds the generated AI_COMPLETE response_format, kept so the
-- contract used for each table is reviewable after the fact.
CREATE OR REPLACE TABLE {{SCHEMA_FQ}}.DISCOVERED_SCHEMAS (
  file_name          VARCHAR,
  table_ordinal      INT,
  table_key          VARCHAR,
  title              VARCHAR,
  columns            ARRAY,
  column_types       ARRAY,
  multi_level_header BOOLEAN,
  header_groups      ARRAY,
  row_count          INT,
  unit_qualifier     VARCHAR,
  target_table       VARCHAR,
  response_schema    VARIANT,
  discovered_at      TIMESTAMP_LTZ
);

In [ ]:
# 1c. Load discovery into the registry, generating the response_format that
# Phase 2 will use for each table.
#
# This is Python rather than SQL because the schema is only known at runtime --
# there is no fixed set of tables to write SQL against.
#
# Column names are computed ONCE here and stored on the registry as
# "field_names". Phase 2 and Phase 3 consume them rather than recomputing, so
# the JSON schema, the extraction SQL and the validation assertions cannot
# drift apart.
#
# Simplifying assumption for this document: discovered headers are already
# clean identifiers and reported types are already INT/VARCHAR, so no
# normalisation helpers are needed. A messier document would need them back.
import json

JSON_TYPE = {"INT": "integer", "FLOAT": "number"}

INSERT_SQL = f"""
    INSERT INTO {SCHEMA_FQ}.DISCOVERED_SCHEMAS
      (file_name, table_ordinal, table_key, title, columns, column_types,
       multi_level_header, header_groups, row_count, unit_qualifier,
       target_table, response_schema, discovered_at)
    SELECT ?, ?, ?, ?,
           PARSE_JSON(?), PARSE_JSON(?),
           ?, PARSE_JSON(?), ?, ?,
           ?, PARSE_JSON(?), CURRENT_TIMESTAMP()
"""


def build_response_schema(keys, types) -> dict:
    """Row-oriented response_format: one object per data row.

    Row-oriented rather than columnar means each row arrives as a self-contained
    object, so there is no index-matched reassembly and therefore no way for a
    dropped value to silently shift a column out of alignment.
    """
    props = {k: {"type": JSON_TYPE.get(t, "string")} for k, t in zip(keys, types)}
    item  = {"type": "object", "properties": props, "required": keys}
    return {"type": "json",
            "schema": {"type": "object",
                       "properties": {"rows": {"type": "array", "items": item}},
                       "required": ["rows"]}}


rows = session.sql(f"""
    SELECT file_name, table_ordinal, table_key, title,
           columns, column_types, multi_level_header,
           header_groups, row_count, unit_qualifier
    FROM {SCHEMA_FQ}.DESCRIBED_TABLES
    ORDER BY file_name, table_ordinal
""").collect()

registry = []
for r in rows:
    cols   = json.loads(r["COLUMNS"])
    types  = json.loads(r["COLUMN_TYPES"])
    groups = json.loads(r["HEADER_GROUPS"])

    # Qualify multi-level headers with their spanning group, then suffix any
    # remaining duplicates, so every column yields a distinct identifier.
    # Padding groups lets zip cover tables with no spanning header; a blank
    # group falls out of split() on its own.
    keys, seen = [], {}
    for c, g in zip(cols, groups + [""] * len(cols)):
        n = "_".join(f"{g} {c}".split()).lower()
        seen[n] = seen.get(n, 0) + 1
        keys.append(n if seen[n] == 1 else f"{n}_{seen[n]}")

    key = r["TABLE_KEY"] or f"table_{r['TABLE_ORDINAL']}"

    registry.append({
        "file_name":          r["FILE_NAME"],
        "table_ordinal":      int(r["TABLE_ORDINAL"]),
        "table_key":          key,
        "title":              r["TITLE"],
        "columns":            cols,
        "column_types":       types,
        "field_names":        keys,
        "multi_level_header": bool(r["MULTI_LEVEL_HEADER"]),
        "header_groups":      groups,
        "row_count":          int(r["ROW_COUNT"]) if r["ROW_COUNT"] is not None else None,
        "unit_qualifier":     r["UNIT_QUALIFIER"],
        "target_table":       key.upper(),
        "response_schema":    build_response_schema(keys, types),
    })

session.sql(f"DELETE FROM {SCHEMA_FQ}.DISCOVERED_SCHEMAS").collect()

for rec in registry:
    session.sql(INSERT_SQL, params=[
        rec["file_name"], rec["table_ordinal"], rec["table_key"], rec["title"],
        json.dumps(rec["columns"]), json.dumps(rec["column_types"]),
        rec["multi_level_header"], json.dumps(rec["header_groups"]),
        rec["row_count"], rec["unit_qualifier"],
        rec["target_table"], json.dumps(rec["response_schema"]),
    ]).collect()

print(f"registered {len(registry)} table(s)\n")
for rec in registry:
    flags = []
    if rec["multi_level_header"]:
        flags.append("multi-level header")
    if rec["unit_qualifier"]:
        flags.append(f"units: {rec['unit_qualifier']}")
    note = f"  [{', '.join(flags)}]" if flags else ""
    print(f"  {rec['target_table']:<30} "
          f"{len(rec['columns'])} cols x {rec['row_count']} rows{note}")
    print(f"      {', '.join(rec['field_names'])}")

In [ ]:
-- 1c. Review the registry before extracting anything.
SELECT table_ordinal, target_table, title,
       ARRAY_SIZE(columns) AS n_columns, row_count,
       multi_level_header, unit_qualifier, columns
FROM {{SCHEMA_FQ}}.DISCOVERED_SCHEMAS
ORDER BY file_name, table_ordinal

---

## Phase 2 — Extract

Every table is extracted with a schema that came from Phase 1, never a guess.

**Method:** `AI_COMPLETE` against the parsed markdown, with a **row-oriented** `response_format` — one JSON object per data row rather than one array per column.

Why row-oriented matters: columnar output has to be reassembled by array position, and if the model drops a value from the middle of one column, every later value shifts up one slot. The result is plausible numbers attached to the wrong keys, with no error raised. Row objects bind each value to its key, so `required` in the schema catches the problem instead of hiding it.

Because Phase 1 already parsed the document, this phase costs **no additional document-processing call** — only a text completion.

This phase is Python-driven because the set of tables is not known until discovery completes; there is no fixed list to write SQL cells against.

In [ ]:
# Phase 2 dispatcher. Reads the registry, generates one AI_COMPLETE statement
# per discovered table, executes it, and records the outcome.
# A failure on one table does not abort the rest.
#
# Column names come from rec["field_names"], computed in cell 1c -- they are
# never recomputed here, so the prompt, the JSON schema and the SELECT list
# always agree.


def q(s) -> str:
    """Single-quote a SQL string literal."""
    return "'" + str(s).replace("'", "''") + "'"


def dq(s) -> str:
    """Double-quote a VARIANT path key, for keys with spaces or mixed case."""
    return '"' + str(s).replace('"', '\\"') + '"'


def py_to_sql_object(obj) -> str:
    """Render a Python dict/list as Snowflake object/array literal syntax."""
    if isinstance(obj, dict):
        inner = ", ".join(f"{q(k)}: {py_to_sql_object(v)}" for k, v in obj.items())
        return "{" + inner + "}"
    if isinstance(obj, list):
        return "[" + ", ".join(py_to_sql_object(v) for v in obj) + "]"
    if isinstance(obj, bool):
        return "TRUE" if obj else "FALSE"
    if obj is None:
        return "NULL"
    if isinstance(obj, (int, float)):
        return str(obj)
    return q(obj)


def build_prompt(rec) -> str:
    """Extraction instruction, enriched with whatever discovery reported."""
    keys = rec["field_names"]
    hint = ""
    if rec["unit_qualifier"]:
        hint += (f"Values are expressed {rec['unit_qualifier']}; return them exactly "
                 f"as printed, without rescaling. ")
    if rec["multi_level_header"] and rec["header_groups"]:
        hint += (f"The header is multi-level with the spanning groups "
                 f"{', '.join(g for g in rec['header_groups'] if g)}. Attribute each "
                 f"column to its correct group -- the group labels sit above a second "
                 f"header row, so match them by column position. Each key below is "
                 f"prefixed with its group name. ")
    return (
        f'Extract the table titled "{rec["title"]}" from the markdown document below. '
        f"{hint}"
        f"Return one object per data row, excluding every header row. "
        f"Use exactly these keys, in this order: {', '.join(keys)}."
    )


def build_extraction_sql(rec) -> str:
    """Generate the CREATE TABLE AS for one discovered table."""
    keys, types = rec["field_names"], rec["column_types"]
    selects = ",\n".join(
        f"  x.VALUE:{dq(keys[i])}::{types[i]} AS {keys[i].upper()}"
        for i in range(len(keys))
    )
    return f"""
CREATE OR REPLACE TABLE {SCHEMA_FQ}.{rec['target_table']} AS
WITH src AS (
  SELECT content AS md
  FROM {SCHEMA_FQ}.PARSED_MARKDOWN
  WHERE file_name = {q(rec['file_name'])}
),
resp AS (
  SELECT AI_COMPLETE(
    model  => {q(MODEL)},
    prompt => {q(build_prompt(rec))} || CHR(10) || CHR(10) || md,
    model_parameters => {{'temperature': 0}},
    response_format  => {py_to_sql_object(rec['response_schema'])}
  ) AS r
  FROM src
)
SELECT
{selects}
FROM resp, LATERAL FLATTEN(input => TRY_PARSE_JSON(r):rows) x
""".strip()


results = []
for rec in registry:
    if not rec["field_names"]:
        results.append({"table": rec["target_table"], "status": "SKIPPED",
                        "rows": None, "expected": rec["row_count"],
                        "error": "discovery reported no columns", "sql": None})
        continue

    stmt = build_extraction_sql(rec)
    try:
        session.sql(stmt).collect()
        n = session.sql(
            f"SELECT COUNT(*) AS n FROM {SCHEMA_FQ}.{rec['target_table']}"
        ).collect()[0]["N"]
        results.append({"table": rec["target_table"], "status": "ok", "rows": n,
                        "expected": rec["row_count"], "error": None, "sql": stmt})
    except Exception as exc:
        results.append({"table": rec["target_table"], "status": "FAILED", "rows": None,
                        "expected": rec["row_count"], "error": str(exc)[:300],
                        "sql": stmt})

print(f"{'table':<30} {'status':<8} {'rows':>5} {'exp':>5}")
print("-" * 52)
for r in results:
    flag = "" if r["rows"] == r["expected"] or r["rows"] is None else "  <- count differs"
    print(f"{r['table']:<30} {r['status']:<8} "
          f"{str(r['rows'] if r['rows'] is not None else '-'):>5} "
          f"{str(r['expected'] if r['expected'] is not None else '-'):>5}{flag}")

failures = [r for r in results if r["status"] != "ok"]
if failures:
    print("\n--- failures ---")
    for r in failures:
        print(f"\n{r['table']}: {r['error']}")

In [ ]:
-- Phase 2. What actually got created.
SELECT table_name, row_count, bytes, created
FROM {{DB}}.INFORMATION_SCHEMA.TABLES
WHERE table_schema = '{{SCHEMA}}'
  AND table_name NOT IN ('PARSED_DOCS', 'DESCRIBED_TABLES', 'DISCOVERED_SCHEMAS')
ORDER BY table_name

In [ ]:
SELECT
  d.table_ordinal                        AS "#",
  d.target_table                         AS "Table",
  c.cols                                 AS "Columns",
  t.row_count                            AS "Rows"
-- FROM DOCUMENT_EXTRACTION.PDF.DISCOVERED_SCHEMAS d
FROM {{SCHEMA_FQ}}.DISCOVERED_SCHEMAS d
-- JOIN DOCUMENT_EXTRACTION.INFORMATION_SCHEMA.TABLES t
JOIN {{DB}}.INFORMATION_SCHEMA.TABLES t
  ON  t.table_schema = 'PDF'
  AND t.table_name   = d.target_table
JOIN (
  SELECT table_name,
         LISTAGG(column_name, ', ') WITHIN GROUP (ORDER BY ordinal_position) AS cols
  -- FROM DOCUMENT_EXTRACTION.INFORMATION_SCHEMA.COLUMNS
  FROM {{DB}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'PDF'
  GROUP BY table_name
) c ON c.table_name = d.target_table
ORDER BY d.table_ordinal;

In [ ]:
select * from {{SCHEMA_FQ}}.enrollment_by_student_level_and_campus;

---

## Phase 3 — Validate

Check that the PDF tables are extracted properly

In [ ]:
# 3a. Deterministic assertion harness. Zero inference cost.
# Every check is expressed so that a PASS means "nothing to report".
#
# Column identifiers come from rec["field_names"] (cell 1c), uppercased the same
# way the dispatcher aliased them -- so these assertions address exactly the
# columns Phase 2 created, with no independent recomputation to drift from.

assertions = []

for rec in registry:
    tbl  = rec["target_table"]
    cols = [k.upper() for k in rec["field_names"]]
    if not cols:
        continue

    # does the table exist at all?
    exists = session.sql(f"""
        SELECT COUNT(*) AS n FROM {DB}.INFORMATION_SCHEMA.TABLES
        WHERE table_schema = '{SCHEMA}' AND table_name = '{tbl}'
    """).collect()[0]["N"]
    if not exists:
        assertions.append({"table": tbl, "check": "table_exists",
                           "actual": 0, "expected": 1, "passed": False})
        continue

    actual_rows = session.sql(
        f"SELECT COUNT(*) AS n FROM {SCHEMA_FQ}.{tbl}"
    ).collect()[0]["N"]

    # 1. row count matches what discovery reported.
    #    This compares two independent AI_COMPLETE calls, so it is the main
    #    automatic cross-check left in the single-method pipeline.
    if rec["row_count"] is not None:
        assertions.append({
            "table": tbl, "check": "row_count_matches_discovery",
            "actual": actual_rows, "expected": rec["row_count"],
            "passed": actual_rows == rec["row_count"],
        })

    # 2. table is not empty
    assertions.append({"table": tbl, "check": "not_empty",
                       "actual": actual_rows, "expected": ">0",
                       "passed": actual_rows > 0})

    # 3. no nulls in the first (key) column
    n_null = session.sql(
        f"SELECT COUNT_IF({cols[0]} IS NULL) AS n FROM {SCHEMA_FQ}.{tbl}"
    ).collect()[0]["N"]
    assertions.append({"table": tbl, "check": f"no_null_key({cols[0]})",
                       "actual": n_null, "expected": 0, "passed": n_null == 0})

    # 4. header text has not leaked into the data.
    #    Compares against the ORIGINAL header strings from discovery, not the
    #    derived identifiers -- leaked values look like the document, not the schema.
    header_literals = ", ".join(
        "'" + str(c).replace("'", "''") + "'" for c in rec["columns"]
    )
    n_leak = session.sql(f"""
        SELECT COUNT(*) AS n FROM {SCHEMA_FQ}.{tbl}
        WHERE {cols[0]}::VARCHAR IN ({header_literals})
    """).collect()[0]["N"]
    assertions.append({"table": tbl, "check": "no_header_leakage",
                       "actual": n_leak, "expected": 0, "passed": n_leak == 0})

    # 5. numeric columns fully coercible
    for i, c in enumerate(cols):
        if rec["column_types"][i] in ("INT", "FLOAT"):
            n_bad = session.sql(f"""
                SELECT COUNT(*) AS n FROM {SCHEMA_FQ}.{tbl}
                WHERE {c} IS NOT NULL
                  AND TRY_CAST({c}::VARCHAR AS {rec['column_types'][i]}) IS NULL
            """).collect()[0]["N"]
            assertions.append({"table": tbl, "check": f"numeric_coercible({c})",
                               "actual": n_bad, "expected": 0, "passed": n_bad == 0})

failed = [a for a in assertions if not a["passed"]]

print(f"{len(assertions) - len(failed)}/{len(assertions)} assertions passed\n")
if failed:
    print(f"{'table':<28} {'check':<34} {'actual':>8} {'expected':>9}")
    print("-" * 82)
    for a in failed:
        print(f"{a['table']:<28} {a['check']:<34} "
              f"{str(a['actual']):>8} {str(a['expected']):>9}")
else:
    print("no failures")

In [ ]:
-- 3a. Find cross-table reconciliation candidates.
-- Tables sharing column names are often summary/breakdown pairs, which gives an
-- exact correctness check for free. Discovery is what makes them findable.
SELECT
  a.target_table                                          AS table_a,
  b.target_table                                          AS table_b,
  ARRAY_INTERSECTION(a.columns, b.columns)                AS shared_columns,
  ARRAY_SIZE(ARRAY_INTERSECTION(a.columns, b.columns))    AS n_shared
FROM {{SCHEMA_FQ}}.DISCOVERED_SCHEMAS a
JOIN {{SCHEMA_FQ}}.DISCOVERED_SCHEMAS b
  ON  a.file_name     = b.file_name
  AND a.table_ordinal < b.table_ordinal
WHERE ARRAY_SIZE(ARRAY_INTERSECTION(a.columns, b.columns)) > 0
ORDER BY n_shared DESC

In [ ]:
# 3a. Cross-table reconciliation -- the strongest check available, and free.
#
# This document presents a combined enrollment table AND per-campus breakdowns
# of the same figures. The breakdowns must therefore reproduce the summary
# exactly. Where a document carries that kind of redundancy, it hands you an
# exact correctness check with no inference, no tolerance and no judgement.
#
# Pairs below come from the reconciliation_candidates result above.

RECONCILE = [
    {
        "summary": "ENROLLMENT_BY_STUDENT_LEVEL_AND_CAMPUS",
        "detail":  "EUGENE_ENROLLMENT_BY_STUDENT_LEVEL_AND_CAMPUS",
        "key":     "TERM",
        "compare": [("EUGENE_UNDERGRADUATE", "UNDERGRADUATE"),
                    ("EUGENE_GRADUATE",      "GRADUATE")],
    },
    {
        "summary": "ENROLLMENT_BY_STUDENT_LEVEL_AND_CAMPUS",
        "detail":  "PORTLAND_ENROLLMENT_BY_STUDENT_LEVEL_AND_CAMPUS",
        "key":     "TERM",
        "compare": [("PORTLAND_UNDERGRADUATE", "UNDERGRADUATE"),
                    ("PORTLAND_GRADUATE",      "GRADUATE")],
    },
]

total_cells = total_bad = 0

for spec in RECONCILE:
    key     = spec["key"]
    pairs   = spec["compare"]
    clauses = " OR ".join(f"s.{a} <> d.{b}" for a, b in pairs)
    selects = ", ".join(f"s.{a} AS summary_{a}, d.{b} AS detail_{b}"
                        for a, b in pairs)

    joined = f"""
        FROM {SCHEMA_FQ}.{spec['summary']} s
        JOIN {SCHEMA_FQ}.{spec['detail']}  d ON s.{key} = d.{key}
    """

    n_rows = session.sql(f"SELECT COUNT(*) AS n {joined}").collect()[0]["N"]
    bad    = session.sql(
        f"SELECT s.{key}, {selects} {joined} WHERE {clauses}"
    ).collect()

    n_cells      = n_rows * len(pairs)
    total_cells += n_cells
    total_bad   += len(bad)

    status = "PASS" if not bad else f"FAIL ({len(bad)} row(s))"
    print(f"{status:<8} {spec['detail']}")
    print(f"         {n_rows} rows x {len(pairs)} measures = {n_cells} cells "
          f"vs {spec['summary']}")
    for b in bad:
        print(f"         {b}")

print(f"\n{total_cells - total_bad}/{total_cells} cells reconcile exactly "
      f"against the summary table.")

In [ ]:
# 3b. LLM-as-judge, scoped to failures only.
# Uses a prompt object containing the original document, so the judge sees
# ground truth that a markdown-mediated extractor never had.
#
# Per the design doc, prompt-object calls cannot be schema-enforced, so this
# returns free text. Pass it through a second single-string AI_COMPLETE with
# response_format if you need structured verdicts.
import json

suspect = sorted({a["table"] for a in assertions if not a["passed"]})

if not suspect:
    print("No assertion failures. Nothing for the judge to look at.")
    print("This is the intended outcome -- the judge is a fallback, not a routine step.")
else:
    print(f"judging {len(suspect)} table(s): {', '.join(suspect)}\n")
    for tbl in suspect:
        rec = next(r for r in registry if r["target_table"] == tbl)
        rows = session.sql(f"SELECT * FROM {SCHEMA_FQ}.{tbl} LIMIT 30").collect()
        rows_json = json.dumps([dict(r.as_dict()) for r in rows], default=str)

        instruction = (
            f'Document {{0}} contains a table titled "{rec["title"]}". '
            f"Our extraction produced these rows:"
        )
        closing = (
            "Compare against the document. Report any cell where the value or its "
            "header attribution is wrong, and say whether we extracted the intended "
            "table at all. If every row matches, reply exactly: MATCH."
        )

        verdict = session.sql(
            """
            SELECT AI_COMPLETE(?, PROMPT(? || CHR(10) || ? || CHR(10) || ?,
                                         TO_FILE(?, ?))) AS judgment
            """,
            params=[MODEL, instruction, rows_json, closing,
                    f"@{STAGE_FQ}", rec["file_name"]],
        ).collect()[0]["JUDGMENT"]

        print(f"--- {tbl} ---")
        print(verdict)
        print()

---

## Summary

Single-method pipeline: `AI_PARSE_DOCUMENT` once per file, then `AI_COMPLETE` for both discovery and extraction.

| Object | Phase | Purpose |
| --- | --- | --- |
| `PARSED_DOCS` | 1a | Raw `AI_PARSE_DOCUMENT` responses, one row per file |
| `PARSED_MARKDOWN` | 1a | View exposing the markdown |
| `DESCRIBED_TABLES` | 1b | LLM-described tables: columns, types, header nesting, units |
| `DISCOVERED_SCHEMAS` | 1c | Registry: generated `response_format` + target name per table |
| *one table per discovered table* | 2 | The extracted data |

### Re-running

- **New documents on the stage** — run from Phase 1 down. `DIRECTORY()` picks them up automatically.
- **Same documents, changed extraction** — Phase 1 output is persisted, so re-run Phase 2 alone. That is the point of persisting both the parse and the discovery.
- **After editing any cell** — the kernel loses `SCHEMA_FQ` and `registry`, so run from the config cell down rather than resuming mid-notebook.

### Where to look first if something fails

1. **Unknown model error** — change `MODEL` in the config cell.
2. **`DIRECTORY()` returns nothing** — the stage needs a directory table and a refresh: `ALTER STAGE <stage> REFRESH`.
3. **Row count differs from discovery** — the two `AI_COMPLETE` calls disagree about what counts as a header row. Compare the extracted rows against `DESCRIBED_TABLES.row_count` and read the markdown for that table.
4. **Wrong values under multi-level headers** — the known weak spot of a markdown-only path. Discovery flags these as `multi_level_header`; if one is wrong, that table is the case for extracting it from the document itself with `AI_EXTRACT` instead.

### What was dropped from the dual-method version

Deterministic markdown parsing (`MD_LINES`, `DETECTED_TABLES`), the cross-method discovery reconciliation, and the path-to-path extraction diff. Those provided free cross-checks; with a single method there is nothing to check against, so Phase 3's SQL assertions and the source document itself carry the full verification load.